In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# =========================================================
# 1) CAMINHOS (compatível com .py e Jupyter)
# =========================================================
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

DATA_PATH = os.path.join(BASE_DIR, "Clean_Dataset.csv")
OUT_DIR = os.path.join(BASE_DIR, "data_gold_flights")
os.makedirs(OUT_DIR, exist_ok=True)

STEP1_PATH = os.path.join(OUT_DIR, "flights_prepared_step1.csv")
TRAIN_PATH = os.path.join(OUT_DIR, "gold_train.csv")
VAL_PATH   = os.path.join(OUT_DIR, "gold_validation.csv")
TEST_PATH  = os.path.join(OUT_DIR, "gold_test.csv")
MAP_PATH   = os.path.join(OUT_DIR, "mappings.json")
SCALER_PATH = os.path.join(OUT_DIR, "scaler_stats.npz")

RANDOM_STATE = 42

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Não encontrei o ficheiro: {DATA_PATH}")

# =========================================================
# 2) LOAD + LIMPEZA BÁSICA
# =========================================================
df = pd.read_csv(DATA_PATH)
print("Shape inicial:", df.shape)

# remover coluna lixo
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# target numérico
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["price"])

# =========================================================
# 3) duration -> duration_min (robusto)
# =========================================================
def duration_to_minutes(x):
    if pd.isna(x):
        return np.nan

    # caso numérico (ex: 2.17 horas)
    if isinstance(x, (int, float, np.integer, np.floating)):
        return int(round(float(x) * 60))

    # caso string (ex: "2h 50m")
    s = str(x).strip().lower()
    h = 0
    m = 0
    mh = re.search(r"(\d+)\s*h", s)
    mm = re.search(r"(\d+)\s*m", s)
    if mh:
        h = int(mh.group(1))
    if mm:
        m = int(mm.group(1))
    total = 60 * h + m
    return total if total > 0 else np.nan

if "duration" not in df.columns:
    raise ValueError("A coluna 'duration' não existe no dataset.")

df["duration_min"] = df["duration"].apply(duration_to_minutes)
print("Duration_min NaN:", df["duration_min"].isna().sum())
df = df.dropna(subset=["duration_min"])

# days_left numérico
df["days_left"] = pd.to_numeric(df["days_left"], errors="coerce")
df = df.dropna(subset=["days_left"])

# =========================================================
# 4) Seleção das colunas relevantes
# =========================================================
needed_cols = [
    "airline",
    "source_city",
    "destination_city",
    "departure_time",
    "arrival_time",
    "stops",
    "class",
    "duration_min",
    "days_left",
    "price"
]
missing = [c for c in needed_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltam colunas: {missing}")

df = df[needed_cols].copy()

# guardar step1 (opcional mas útil)
df.to_csv(STEP1_PATH, index=False)


# =========================================================
# 5) Encoding categóricas -> índices (para embeddings)
# =========================================================
cat_cols = [
    "airline",
    "source_city",
    "destination_city",
    "departure_time",
    "arrival_time",
    "stops",
    "class"
]
mappings = {}

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()
    uniques = sorted(df[col].unique().tolist())

    # 0 reservado para UNK (se aparecer algo fora do treino)
    mapping = {"UNK": 0}
    for i, v in enumerate(uniques, start=1):
        mapping[v] = i

    mappings[col] = mapping
    df[col + "_idx"] = df[col].map(mapping).fillna(0).astype("int32")

# =========================================================
# 6) Normalização das numéricas
# =========================================================
num_cols = ["duration_min", "days_left"]
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols].values.astype("float32"))

np.savez(
    SCALER_PATH,
    mean=scaler.mean_.astype("float32"),
    scale=scaler.scale_.astype("float32"),
    num_cols=np.array(num_cols)
)
with open(MAP_PATH, "w", encoding="utf-8") as f:
    json.dump(mappings, f, ensure_ascii=False, indent=2)



# =========================================================
# 7) GOLD final + split (80/10/10)
# =========================================================
gold = df[
    [
        "airline_idx",
        "source_city_idx",
        "destination_city_idx",
        "departure_time_idx",
        "arrival_time_idx",
        "stops_idx",
        "class_idx",
        "duration_min",
        "days_left",
        "price",
    ]
].copy()

train_df, temp_df = train_test_split(gold, test_size=0.2, random_state=RANDOM_STATE)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=RANDOM_STATE)

train_df.to_csv(TRAIN_PATH, index=False)
val_df.to_csv(VAL_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)


print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)


print(TRAIN_PATH)
print(VAL_PATH)
print(TEST_PATH)

# =========================================================
# 8) (info) vocab sizes para embeddings
# =========================================================
vocab_sizes = {col: (max(mappings[col].values()) + 1) for col in mappings}
print("\nVocab sizes:")
for k, v in vocab_sizes.items():
    print(f"{k:>18}: {v}")


Shape inicial: (300153, 12)
Duration_min NaN: 0
Train: (240122, 10)
Val  : (30015, 10)
Test : (30016, 10)
/Users/eduardaferreira/Desktop/projeto sistemas /archive-2/data_gold_flights/gold_train.csv
/Users/eduardaferreira/Desktop/projeto sistemas /archive-2/data_gold_flights/gold_validation.csv
/Users/eduardaferreira/Desktop/projeto sistemas /archive-2/data_gold_flights/gold_test.csv

Vocab sizes:
           airline: 7
       source_city: 7
  destination_city: 7
    departure_time: 7
      arrival_time: 7
             stops: 4
             class: 3


In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from keras import layers, models, callbacks
 
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

GOLD_DIR = os.path.join(BASE_DIR, "data_gold_flights")
TRAIN_PATH = os.path.join(GOLD_DIR, "gold_train.csv")
VAL_PATH   = os.path.join(GOLD_DIR, "gold_validation.csv")

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"Não encontrei: {TRAIN_PATH}")
if not os.path.exists(VAL_PATH):
    raise FileNotFoundError(f"Não encontrei: {VAL_PATH}")

BATCH_SIZE = 256
EPOCHS = 10

# 2) VOCAB SIZES (para embeddings) — lidos do train

tmp = pd.read_csv(TRAIN_PATH, nrows=200000)

vocab_sizes = {
    "airline": int(tmp["airline_idx"].max() + 1),
    "source_city": int(tmp["source_city_idx"].max() + 1),
    "destination_city": int(tmp["destination_city_idx"].max() + 1),
    "departure_time": int(tmp["departure_time_idx"].max() + 1),
    "arrival_time": int(tmp["arrival_time_idx"].max() + 1),
    "stops": int(tmp["stops_idx"].max() + 1),
    "class": int(tmp["class_idx"].max() + 1),
}
print("Vocab sizes:", vocab_sizes)


# 3) GERADOR DE DADOS (eficiência de memória)

def data_generator(file_path, batch_size=256):
    while True:
        for chunk in pd.read_csv(file_path, chunksize=batch_size):
            X = {
                "airline_in": chunk["airline_idx"].values.astype("int32"),
                "source_city_in": chunk["source_city_idx"].values.astype("int32"),
                "destination_city_in": chunk["destination_city_idx"].values.astype("int32"),
                "departure_time_in": chunk["departure_time_idx"].values.astype("int32"),
                "arrival_time_in": chunk["arrival_time_idx"].values.astype("int32"),
                "stops_in": chunk["stops_idx"].values.astype("int32"),
                "class_in": chunk["class_idx"].values.astype("int32"),
                "numeric_in": chunk[["duration_min", "days_left"]].values.astype("float32"),
            }
            y = chunk["price"].values.astype("float32")
            yield (X, y)


output_sig = (
    {
        "airline_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "source_city_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "destination_city_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "departure_time_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "arrival_time_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "stops_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "class_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "numeric_in": tf.TensorSpec(shape=(None, 2), dtype=tf.float32),
    },
    tf.TensorSpec(shape=(None,), dtype=tf.float32),
)

train_ds = tf.data.Dataset.from_generator(
    lambda: data_generator(TRAIN_PATH, BATCH_SIZE),
    output_signature=output_sig
).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(
    lambda: data_generator(VAL_PATH, BATCH_SIZE),
    output_signature=output_sig
).prefetch(tf.data.AUTOTUNE)


# 4) REDE NEURONAL (Functional API, multi-input + embeddings)

def make_embedding_input(name, vocab_size, embed_dim):
    inp = layers.Input(shape=(), dtype="int32", name=name)
    emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)(inp)
    emb = layers.Flatten()(emb)
    return inp, emb

def build_model(vocab_sizes):
    inp_airline, x_airline = make_embedding_input("airline_in", vocab_sizes["airline"], 4)
    inp_sc, x_sc = make_embedding_input("source_city_in", vocab_sizes["source_city"], 4)
    inp_dc, x_dc = make_embedding_input("destination_city_in", vocab_sizes["destination_city"], 4)
    inp_dep, x_dep = make_embedding_input("departure_time_in", vocab_sizes["departure_time"], 3)
    inp_arr, x_arr = make_embedding_input("arrival_time_in", vocab_sizes["arrival_time"], 3)
    inp_stops, x_stops = make_embedding_input("stops_in", vocab_sizes["stops"], 2)
    inp_class, x_class = make_embedding_input("class_in", vocab_sizes["class"], 2)

    inp_num = layers.Input(shape=(2,), dtype="float32", name="numeric_in")

    concat = layers.Concatenate()([x_airline, x_sc, x_dc, x_dep, x_arr, x_stops, x_class, inp_num])

    x = layers.Dense(128, activation="relu")(concat)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dense(32, activation="relu")(x)

    out = layers.Dense(1, activation="linear", name="price_output")(x)

    model = models.Model(
        inputs=[inp_airline, inp_sc, inp_dc, inp_dep, inp_arr, inp_stops, inp_class, inp_num],
        outputs=out
    )
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

model = build_model(vocab_sizes)
model.summary()


# 5) TREINO (com callbacks)

my_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    callbacks.ModelCheckpoint(os.path.join(GOLD_DIR, "best_model_flights.keras"), save_best_only=True),
]


n_train = sum(1 for _ in open(TRAIN_PATH)) - 1
n_val   = sum(1 for _ in open(VAL_PATH)) - 1

steps_per_epoch = max(1, n_train // BATCH_SIZE)
validation_steps = max(1, n_val // BATCH_SIZE)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=my_callbacks
)

print(os.path.join(GOLD_DIR, "best_model_flights.keras"))


Vocab sizes: {'airline': 7, 'source_city': 7, 'destination_city': 7, 'departure_time': 7, 'arrival_time': 7, 'stops': 4, 'class': 3}


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ airline_in          │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ source_city_in      │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ destination_city_in │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ departure_time_in   │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ arrival_time_in     │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stops_in            │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_in            │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 4)         │         28 │ airline_in[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 4)         │         28 │ source_city_in[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 4)         │         28 │ destination_city… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 3)         │         21 │ departure_time_i… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 3)         │         21 │ arrival_time_in[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 2)         │          8 │ stops_in[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 2)         │          6 │ class_in[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 4)         │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 4)         │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 4)         │          0 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 3)         │          0 │ embedding_3[0][0

 Total params: 13,709 (53.55 KB)

 Trainable params: 13,709 (53.55 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 197774592.0000 - mae: 7935.2554 - val_loss: 34779668.0000 - val_mae: 3672.4070
Epoch 2/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 35656028.0000 - mae: 3760.0874 - val_loss: 30426904.0000 - val_mae: 3386.3308
Epoch 3/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 33185500.0000 - mae: 3553.3962 - val_loss: 29379868.0000 - val_mae: 3264.4944
Epoch 4/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 32408292.0000 - mae: 3468.2983 - val_loss: 28736534.0000 - val_mae: 3209.5122
Epoch 5/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 31955516.0000 - mae: 3421.7913 - val_loss: 28410552.0000 - val_mae: 3183.1062
Epoch 6/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 31643720.0000 - mae: 3395.6443 - val_loss: 28167982.0000 - val_mae: 3161.2249
Epoch 7/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 31371066.0000 - mae: 3366.4673 - val_loss: 28027014.0000 - val_mae: 3138.1641
Epoch 8/10
937/937 ━━━━━━━━━━━━━━

In [ ]:
####### MODELO 2 - Dropout 0.5 #######
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from keras import layers, models, callbacks


try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

GOLD_DIR = os.path.join(BASE_DIR, "data_gold_flights")
TRAIN_PATH = os.path.join(GOLD_DIR, "gold_train.csv")
VAL_PATH   = os.path.join(GOLD_DIR, "gold_validation.csv")

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"Não encontrei: {TRAIN_PATH}")
if not os.path.exists(VAL_PATH):
    raise FileNotFoundError(f"Não encontrei: {VAL_PATH}")

BATCH_SIZE = 256
EPOCHS = 10

# 2) VOCAB SIZES (para embeddings)

tmp = pd.read_csv(TRAIN_PATH, nrows=200000)

vocab_sizes = {
    "airline": int(tmp["airline_idx"].max() + 1),
    "source_city": int(tmp["source_city_idx"].max() + 1),
    "destination_city": int(tmp["destination_city_idx"].max() + 1),
    "departure_time": int(tmp["departure_time_idx"].max() + 1),
    "arrival_time": int(tmp["arrival_time_idx"].max() + 1),
    "stops": int(tmp["stops_idx"].max() + 1),
    "class": int(tmp["class_idx"].max() + 1),
}

print("Vocab sizes:", vocab_sizes)


# 3) GERADOR DE DADOS (eficiência de memória)
def data_generator(file_path, batch_size=256):
    while True:
        for chunk in pd.read_csv(file_path, chunksize=batch_size):
            X = {
                "airline_in": chunk["airline_idx"].values.astype("int32"),
                "source_city_in": chunk["source_city_idx"].values.astype("int32"),
                "destination_city_in": chunk["destination_city_idx"].values.astype("int32"),
                "departure_time_in": chunk["departure_time_idx"].values.astype("int32"),
                "arrival_time_in": chunk["arrival_time_idx"].values.astype("int32"),
                "stops_in": chunk["stops_idx"].values.astype("int32"),
                "class_in": chunk["class_idx"].values.astype("int32"),
                "numeric_in": chunk[["duration_min", "days_left"]].values.astype("float32"),
            }
            y = chunk["price"].values.astype("float32")
            yield (X, y)


output_sig = (
    {
        "airline_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "source_city_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "destination_city_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "departure_time_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "arrival_time_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "stops_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "class_in": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "numeric_in": tf.TensorSpec(shape=(None, 2), dtype=tf.float32),
    },
    tf.TensorSpec(shape=(None,), dtype=tf.float32),
)

train_ds = tf.data.Dataset.from_generator(
    lambda: data_generator(TRAIN_PATH, BATCH_SIZE),
    output_signature=output_sig
).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(
    lambda: data_generator(VAL_PATH, BATCH_SIZE),
    output_signature=output_sig
).prefetch(tf.data.AUTOTUNE)


# 4) MODELO 2 — REDE NEURONAL (Dropout = 0.5)

def make_embedding_input(name, vocab_size, embed_dim):
    inp = layers.Input(shape=(), dtype="int32", name=name)
    emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)(inp)
    emb = layers.Flatten()(emb)
    return inp, emb

def build_model(vocab_sizes):
    inp_airline, x_airline = make_embedding_input("airline_in", vocab_sizes["airline"], 4)
    inp_sc, x_sc = make_embedding_input("source_city_in", vocab_sizes["source_city"], 4)
    inp_dc, x_dc = make_embedding_input("destination_city_in", vocab_sizes["destination_city"], 4)
    inp_dep, x_dep = make_embedding_input("departure_time_in", vocab_sizes["departure_time"], 3)
    inp_arr, x_arr = make_embedding_input("arrival_time_in", vocab_sizes["arrival_time"], 3)
    inp_stops, x_stops = make_embedding_input("stops_in", vocab_sizes["stops"], 2)
    inp_class, x_class = make_embedding_input("class_in", vocab_sizes["class"], 2)

    inp_num = layers.Input(shape=(2,), dtype="float32", name="numeric_in")

    concat = layers.Concatenate()(
        [x_airline, x_sc, x_dc, x_dep, x_arr, x_stops, x_class, inp_num]
    )

    x = layers.Dense(128, activation="relu")(concat)
    x = layers.Dropout(0.5)(x)   #  ALTERAÇÃO DO MODELO 2
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dense(32, activation="relu")(x)

    out = layers.Dense(1, activation="linear", name="price_output")(x)

    model = models.Model(
        inputs=[inp_airline, inp_sc, inp_dc, inp_dep, inp_arr, inp_stops, inp_class, inp_num],
        outputs=out
    )
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

model = build_model(vocab_sizes)
model.summary()


# 5) TREINO 

my_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    callbacks.ModelCheckpoint(
        os.path.join(GOLD_DIR, "best_model_flights_v2.keras"),
        save_best_only=True
    ),
]

n_train = sum(1 for _ in open(TRAIN_PATH)) - 1
n_val   = sum(1 for _ in open(VAL_PATH)) - 1

steps_per_epoch = max(1, n_train // BATCH_SIZE)
validation_steps = max(1, n_val // BATCH_SIZE)

print("\n A iniciar treino MODELO 2 (Dropout = 0.5)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=my_callbacks
)


print(os.path.join(GOLD_DIR, "best_model_flights_v2.keras"))


Vocab sizes: {'airline': 7, 'source_city': 7, 'destination_city': 7, 'departure_time': 7, 'arrival_time': 7, 'stops': 4, 'class': 3}


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ airline_in          │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ source_city_in      │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ destination_city_in │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ departure_time_in   │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ arrival_time_in     │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stops_in            │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_in            │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, 4)         │         28 │ airline_in[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_8         │ (None, 4)         │         28 │ source_city_in[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_9         │ (None, 4)         │         28 │ destination_city… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_10        │ (None, 3)         │         21 │ departure_time_i… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_11        │ (None, 3)         │         21 │ arrival_time_in[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_12        │ (None, 2)         │          8 │ stops_in[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_13        │ (None, 2)         │          6 │ class_in[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_7 (Flatten) │ (None, 4)         │          0 │ embedding_7[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_8 (Flatten) │ (None, 4)         │          0 │ embedding_8[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_9 (Flatten) │ (None, 4)         │          0 │ embedding_9[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_10          │ (None, 3)         │          0 │ embedding_10[0][

 Total params: 13,709 (53.55 KB)

 Trainable params: 13,709 (53.55 KB)

 Non-trainable params: 0 (0.00 B)


 A iniciar treino MODELO 2 (Dropout = 0.5)...
Epoch 1/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 206088560.0000 - mae: 8248.1338 - val_loss: 35769512.0000 - val_mae: 3731.6580
Epoch 2/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 40552556.0000 - mae: 4019.1099 - val_loss: 31158456.0000 - val_mae: 3459.2051
Epoch 3/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 38433336.0000 - mae: 3860.3518 - val_loss: 30247682.0000 - val_mae: 3346.4426
Epoch 4/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 37556116.0000 - mae: 3759.9680 - val_loss: 29386688.0000 - val_mae: 3265.1897
Epoch 5/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 36802680.0000 - mae: 3687.1016 - val_loss: 29018044.0000 - val_mae: 3219.4253
Epoch 6/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 36478416.0000 - mae: 3647.1526 - val_loss: 29235836.0000 - val_mae: 3219.9941
Epoch 7/10
937/937 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 35939268.0000 - mae: 3607.0652 - val_loss: 29186612.0000 - val_m